In [ ]:
# Colab/bootstrap: clone this repository and install it editable.
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/JeonDongJun/mindscopex_analysis"
MARK_REL = Path("src") / "mindscopex_analysis" / "__init__.py"


def find_repo_root(start=None):
    candidate = Path(start or Path.cwd()).resolve()
    for path in [candidate, *candidate.parents]:
        if (path / MARK_REL).is_file():
            return path
    return None


root = find_repo_root()
if root is None:
    workdir = Path(os.environ.get("COLAB_REPO_DIR", "/content/mindscopex_analysis"))
    if (workdir / MARK_REL).is_file():
        subprocess.run(["git", "-C", str(workdir), "pull", "--ff-only"], check=False)
        root = workdir
    else:
        workdir.parent.mkdir(parents=True, exist_ok=True)
        if workdir.exists():
            shutil.rmtree(workdir)
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(workdir)])
        root = workdir

os.environ["MINDSCOPEX_ROOT"] = str(root.resolve())
os.chdir(root)
QWEN35_TRANSFORMERS_REVISION = "b70d02fc724d04c916832ca4ead03ff05e8fb1ee"
qwen35_probe = subprocess.run(
    [sys.executable, "-c", "from transformers import AutoModelForMultimodalLM"],
    capture_output=True,
)
if qwen35_probe.returncode != 0:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        f"transformers @ git+https://github.com/huggingface/transformers.git@{QWEN35_TRANSFORMERS_REVISION}",
        "torchvision", "pillow",
    ])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
print("ready:", root)


# 00. Qwen CRT 실제 텍스트 답변

Qwen 모델군에 CRT 문제를 직접 제시하고, thinking/non-thinking 모드에서 생성되는 전체 텍스트를 확인하는 기준 실험입니다. 두 모드 모두 보이는 최종 출력은 짧은 답만 요구하며, thinking 모드에서는 reasoning이 생성된 `<think>...</think>` 안에만 있는지 함께 검사합니다.

## 실행 전 고려사항

1. 기본 모델은 post-trained `Qwen3.5-2B`, `9B`, `27B`, `35B-A3B`이며 메모리를 비우면서 한 번에 하나만 GPU에 올립니다.
2. 시스템 프롬프트는 정답 예시를 포함하지 않으며, 보이는 최종 응답을 답과 단위만 있는 한 줄로 제한합니다. thinking 모드의 reasoning은 모델 고유 thinking 영역에서만 허용합니다.
3. 샘플링은 [Qwen3.5 공식 컬렉션](https://huggingface.co/collections/Qwen/qwen35)의 모델 설정을 따르고 seed를 기록합니다. 샘플링 결과는 seed에 따라 달라질 수 있습니다.
4. Qwen3.5 chat template가 입력에 미리 넣는 `<think>` 시작 태그는 생성 suffix에 복원한 뒤 검사합니다. `think_protocol_ok=False`이면 누락 태그, 빈 reasoning, 잘림, 최종 답 누락 중 원인을 표시합니다.
5. `answer_only`와 `format_issue`는 thinking protocol과 별개로 최종 응답이 답만 포함하는지 검사합니다.
6. `answer_label`은 최종 답변 텍스트의 단순 문자열 판정입니다. `both`와 `other`는 반드시 원문을 직접 확인합니다.
7. 주 mechanistic 대상은 checkpoint와 공식 SAE가 직접 일치하는 `Qwen3.5-27B`입니다. 2B, 9B, 35B-A3B는 공개 SAE가 Base checkpoint용이므로 행동 모델로의 전이를 주장하지 않고 Base control로 분석합니다.
8. 네 모델 모두 최신 Qwen3.5 지원 Transformers가 필요합니다. 첫 셀은 검증된 revision을 자동 설치하며, 27B와 35B-A3B는 A100 80GB에서도 메모리 여유를 확인해 순차 실행합니다.
9. `RUN_PRESET`은 `pilot`, `hagendorff_smoke`, `hagendorff_full`, `verbal_crt`, `crt7_classic`, `crt2`, `yax_crt_isomorph` 중 선택합니다(`nature_smoke`/`nature_full`은 하위호환 별칭). smoke는 유형별 3문항, full은 Hagendorff et al. (2023)의 CRT 150문항입니다. 모든 데이터셋 목록·통계·출처는 `docs/datasets.md`와 `available_lure_datasets()`로 확인합니다.
10. Hagendorff 전체 세트와 기본 4개 모델·2개 모드·seed 1개를 함께 실행하면 1,200응답입니다. 데이터는 저장소에 커밋된 JSON(`load_lure_dataset`)에서 로드하므로 다운로드가 필요 없습니다. 먼저 `hagendorff_smoke`로 분류·저장 경로를 검증하세요.
11. 공개 CRT 문항은 2023년부터 공개되어 Qwen 학습 데이터 노출 가능성이 있으므로, 최종 연구에서는 별도의 미공개 변형 세트도 함께 사용해야 합니다.
12. protocol 실패와 `both` 응답은 다른 seed로 최대 `MAX_RETRIES`회 다시 생성하며 모든 시도 원문을 JSON에 보존합니다. 그래도 남은 `both/other`는 최종 3분류에서 hallucination에 포함하되 품질 표에서 따로 확인합니다.
13. `GENERATION_PROTOCOL="qwen_native"`는 Qwen 권장 sampling과 answer-only prompt를 사용합니다. `"deterministic_replication"`은 sampling·재시도·system prompt를 끄고 원 Hagendorff 연구에 가까운 행동 기준선을 만듭니다.
14. `truncated=True`이면 reasoning이 끝나지 않은 것이므로 `MAX_NEW_TOKENS`를 늘려 다시 실행합니다.

In [ ]:
import transformers

print({
    "transformers_version": transformers.__version__,
    "qwen35_transformers_revision": QWEN35_TRANSFORMERS_REVISION,
})


In [ ]:
import os
import sys
from pathlib import Path

root = Path(os.environ.get("MINDSCOPEX_ROOT", Path.cwd())).resolve()
if not (root / "src" / "mindscopex_analysis" / "__init__.py").is_file():
    for candidate in [root, *root.parents]:
        if (candidate / "src" / "mindscopex_analysis" / "__init__.py").is_file():
            root = candidate
            break
    else:
        raise RuntimeError("Could not find repository root. Run the clone cell first.")

src_path = str(root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(root)


In [ ]:
from html import escape

from IPython.display import HTML, display

from mindscopex_analysis import (
    CRT_FINAL_ANSWER_SYSTEM_PROMPT,
    DEFAULT_QWEN_CHAT_MODEL_IDS,
    QWEN_LARGE_CHAT_MODEL_IDS,
    RECOMMENDED_INTERPRETABILITY_MODEL_ID,
    RECOMMENDED_INTERPRETABILITY_SAE_REPO_ID,
    available_lure_datasets,
    clear_device_cache,
    generate_crt_response_suite,
    load_qwen_text_generation_model,
    lure_dataset_cases,
    lure_dataset_info,
    recommended_dtype_name,
    save_crt_markdown_report,
    save_qwen_text_responses,
    summarize_crt_accuracy,
    summarize_crt_accuracy_by_family,
)

print("available datasets:", available_lure_datasets())


def display_records(rows, columns):
    head = "".join(f"<th>{escape(str(column))}</th>" for column in columns)
    body = []
    for row in rows:
        cells = "".join(f"<td>{escape(str(row.get(column, '')))}</td>" for column in columns)
        body.append(f"<tr>{cells}</tr>")
    table = (
        "<div style='overflow-x:auto'><table style='border-collapse:collapse'>"
        f"<thead><tr>{head}</tr></thead><tbody>{''.join(body)}</tbody></table></div>"
    )
    display(HTML(table))

In [ ]:
RUN_PRESET = "hagendorff_smoke"  # see RUN_PRESETS below
GENERATION_PROTOCOL = "qwen_native"  # qwen_native, deterministic_replication
SEEDS = (42,)  # sampling 반복 시 예: (42, 1042, 2042, 3042, 4042)
MODEL_IDS_TO_RUN = tuple(DEFAULT_QWEN_CHAT_MODEL_IDS)

# dataset = data/<id>.json (see docs/datasets.md). limit_per_family caps items per family
# (mirrors the old "smoke" idea of 3 per CRT type). None loads every item.
RUN_PRESETS = {
    "pilot": {"dataset": "crt_pilot", "limit_per_family": None},
    "hagendorff_smoke": {"dataset": "hagendorff_crt", "limit_per_family": 3},
    "hagendorff_full": {"dataset": "hagendorff_crt", "limit_per_family": None},
    "verbal_crt": {"dataset": "verbal_crt", "limit_per_family": None},
    "crt7_classic": {"dataset": "crt7_classic", "limit_per_family": None},
    "crt2": {"dataset": "crt2", "limit_per_family": None},
    "yax_crt_isomorph": {"dataset": "yax_crt_isomorph", "limit_per_family": None},
    # Back-compat aliases for the pre-rename preset names.
    "nature_smoke": {"dataset": "hagendorff_crt", "limit_per_family": 3},
    "nature_full": {"dataset": "hagendorff_crt", "limit_per_family": None},
}
PROTOCOLS = {
    "qwen_native": {
        "do_sample": True,
        "system_prompt": CRT_FINAL_ANSWER_SYSTEM_PROMPT,
        "max_retries": 2,
        "retry_protocol_issues": True,
        "retry_both": True,
    },
    "deterministic_replication": {
        "do_sample": False,
        "system_prompt": "",
        "max_retries": 0,
        "retry_protocol_issues": False,
        "retry_both": False,
    },
}

if RUN_PRESET not in RUN_PRESETS:
    raise ValueError(f"Unknown RUN_PRESET={RUN_PRESET!r}. Options: {sorted(RUN_PRESETS)}")
if GENERATION_PROTOCOL not in PROTOCOLS:
    raise ValueError(f"Unknown GENERATION_PROTOCOL={GENERATION_PROTOCOL!r}")
if not SEEDS:
    raise ValueError("SEEDS must contain at least one seed")

preset = RUN_PRESETS[RUN_PRESET]
protocol = PROTOCOLS[GENERATION_PROTOCOL]
model_ids = list(MODEL_IDS_TO_RUN)

MODEL_SPECS = [
    {
        "model_id": model_id,
        "thinking_modes": (False, True),
        "use_chat_template": True,
        "max_new_tokens": 2048 if model_id in QWEN_LARGE_CHAT_MODEL_IDS else None,
    }
    for model_id in model_ids
]

DATASET_NAME = preset["dataset"]
LIMIT_PER_FAMILY = preset["limit_per_family"]
CASES = lure_dataset_cases(DATASET_NAME, limit_per_family=LIMIT_PER_FAMILY)
_dataset_source = lure_dataset_info(DATASET_NAME).source
DATASET_REFERENCE = (
    _dataset_source.get("doi") or _dataset_source.get("project_url") or DATASET_NAME
)

DTYPE = recommended_dtype_name()
MAX_NEW_TOKENS = 4096
MAX_RETRIES = protocol["max_retries"]
RETRY_PROTOCOL_ISSUES = protocol["retry_protocol_issues"]
RETRY_BOTH = protocol["retry_both"]
DO_SAMPLE = protocol["do_sample"]
SYSTEM_PROMPT = protocol["system_prompt"]
seed_label = "-".join(str(seed) for seed in SEEDS)
RUN_LABEL = f"{RUN_PRESET}_{GENERATION_PROTOCOL}_seeds-{seed_label}"
OUTPUT_PATH = root / "outputs" / f"00_qwen_crt_text_responses_{RUN_LABEL}.json"
REPORT_PATH = root / "outputs" / f"00_qwen_crt_text_responses_{RUN_LABEL}.md"
EXPECTED_RESPONSES = (
    sum(len(spec["thinking_modes"]) for spec in MODEL_SPECS) * len(CASES) * len(SEEDS)
)

print({
    "models": [spec["model_id"] for spec in MODEL_SPECS],
    "run_preset": RUN_PRESET,
    "generation_protocol": GENERATION_PROTOCOL,
    "dataset": DATASET_NAME,
    "dataset_reference": DATASET_REFERENCE,
    "n_cases": len(CASES),
    "expected_responses": EXPECTED_RESPONSES,
    "case_ids_preview": [case.case_id for case in CASES[:10]],
    "dtype": DTYPE,
    "max_new_tokens": MAX_NEW_TOKENS,
    "max_retries": MAX_RETRIES,
    "do_sample": DO_SAMPLE,
    "seeds": SEEDS,
    "system_prompt": SYSTEM_PROMPT,
    "recommended_analysis_model": RECOMMENDED_INTERPRETABILITY_MODEL_ID,
    "recommended_sae": RECOMMENDED_INTERPRETABILITY_SAE_REPO_ID,
})
if len(SEEDS) > 1:
    print(
        "Caution: pooled Wilson intervals treat repeated-seed responses as independent; "
        "use an item-clustered bootstrap for paper-level inference."
    )

In [ ]:
CASE_PREVIEW_LIMIT = 12
case_rows = [
    {
        "case": case.case_id,
        "family": case.family,
        "correct": case.correct_answer.strip(),
        "lure": case.lure_answer.strip(),
        "prompt": case.prompt,
    }
    for case in CASES[:CASE_PREVIEW_LIMIT]
]
display_records(case_rows, ["case", "family", "correct", "lure", "prompt"])
if len(CASES) > CASE_PREVIEW_LIMIT:
    print(f"showing {CASE_PREVIEW_LIMIT} of {len(CASES)} cases")


In [ ]:
responses = []

for spec in MODEL_SPECS:
    model_id = spec["model_id"]
    print(f"\nLoading {model_id} ...")
    model, tokenizer = load_qwen_text_generation_model(
        model_id,
        device_map="auto",
        dtype=DTYPE,
    )

    for seed in SEEDS:
        print(f"Running seed {seed} ...")
        model_responses = generate_crt_response_suite(
            model,
            tokenizer,
            CASES,
            model_id=model_id,
            thinking_modes=spec["thinking_modes"],
            use_chat_template=spec["use_chat_template"],
            system_prompt=SYSTEM_PROMPT,
            max_new_tokens=spec.get("max_new_tokens") or MAX_NEW_TOKENS,
            do_sample=DO_SAMPLE,
            seed=seed,
            max_retries=MAX_RETRIES,
            retry_protocol_issues=RETRY_PROTOCOL_ISSUES,
            retry_both=RETRY_BOTH,
        )
        responses.extend(model_responses)
        save_qwen_text_responses(responses, OUTPUT_PATH)
        save_crt_markdown_report(
            responses,
            REPORT_PATH,
            dataset_name=f"{DATASET_NAME}:{GENERATION_PROTOCOL}",
            dataset_reference=DATASET_REFERENCE,
        )

        for response in model_responses:
            preview = response.answer.replace("\n", " ")[:120]
            print(
                f"[{response.case_id} | {response.mode} | seed={response.seed} | "
                f"{response.evaluation_label} | attempts={response.generation_attempts}] {preview}"
            )
        del model_responses

    del model, tokenizer
    clear_device_cache()

print(f"\nsaved {len(responses)} responses to {OUTPUT_PATH}")
print(f"saved readable summary to {REPORT_PATH}")


In [ ]:
summary_rows = []
for response in responses:
    row = response.summary_row()
    row["final_answer"] = row["final_answer"].replace("\n", " ")[:200]
    summary_rows.append(row)

display_records(
    summary_rows,
    [
        "model", "case", "mode", "think_block", "reasoning_chars",
        "think_protocol_ok", "protocol_issue", "answer_only",
        "format_issue", "label", "outcome", "final_answer",
        "attempts", "retry_reasons", "output_tokens",
        "seconds", "truncated",
    ],
)


In [ ]:
SELECT_MODEL = ""  # 예: Qwen3.5-9B, 빈 문자열이면 전체
SELECT_CASE = ""   # 예: bat_ball_original, 빈 문자열이면 전체
SHOW_THINKING = False  # True로 바꾸면 native thinking 블록을 점검할 수 있습니다.

selected = [
    response
    for response in responses
    if (not SELECT_MODEL or response.model_id.endswith(SELECT_MODEL))
    and (not SELECT_CASE or response.case_id == SELECT_CASE)
]

for response in selected:
    heading = (
        f"{response.model_id} | {response.case_id} | "
        f"{response.mode} | {response.answer_label}"
    )
    sections = [f"<h3>{escape(heading)}</h3>"]
    protocol = (
        f"thinking block: {response.has_thinking_block}, "
        f"reasoning chars: {len(response.thinking)}, "
        f"protocol ok: {response.thinking_protocol_ok}, "
        f"protocol issue: {response.thinking_protocol_issue}, "
        f"answer only: {response.final_answer_format_ok}, "
        f"format issue: {response.final_answer_format_issue}"
    )
    sections.append(f"<p>{escape(protocol)}</p>")
    if SHOW_THINKING and response.thinking:
        sections.append(f"<h4>Thinking</h4><pre>{escape(response.thinking)}</pre>")
    sections.append(f"<h4>Final answer</h4><pre>{escape(response.answer)}</pre>")
    display(HTML("".join(sections)))


In [ ]:
accuracy_rows = summarize_crt_accuracy(responses)
family_accuracy_rows = summarize_crt_accuracy_by_family(responses)
display_records(
    accuracy_rows,
    [
        "model", "mode", "total", "correct", "lure", "hallucination",
        "accuracy", "retried_responses", "retry_attempts", "both", "other",
        "format_failures", "protocol_failures",
    ],
)

family_display_rows = [
    {
        **row,
        "accuracy": f"{row['accuracy']:.1%}",
        "lure_rate": f"{row['lure_rate']:.1%}",
        "lure_rate_95ci": (
            f"[{row['lure_rate_ci_low']:.1%}, {row['lure_rate_ci_high']:.1%}]"
        ),
    }
    for row in family_accuracy_rows
]
display_records(
    family_display_rows,
    [
        "model", "mode", "family", "total", "correct", "lure",
        "hallucination", "accuracy", "lure_rate", "lure_rate_95ci",
    ],
)

protocol_rows = [
    {
        "model": response.model_id.rsplit("/", 1)[-1],
        "case": response.case_id,
        "mode": response.mode,
        "think_block": response.has_thinking_block,
        "reasoning_chars": len(response.thinking),
        "protocol_ok": response.thinking_protocol_ok,
        "protocol_issue": response.thinking_protocol_issue,
        "answer_only": response.final_answer_format_ok,
        "format_issue": response.final_answer_format_issue,
        "outcome": response.evaluation_label,
        "attempts": response.generation_attempts,
        "retry_reasons": ", ".join(response.retry_reasons),
        "truncated": response.hit_max_tokens,
        "final_words": len(response.answer.split()),
        "final_answer": response.answer.replace("\n", " ")[:120],
        "raw_preview": response.raw_text.replace("\n", " ")[:240],
    }
    for response in responses
]
display_records(
    protocol_rows,
    [
        "model", "case", "mode", "think_block", "reasoning_chars",
        "protocol_ok", "protocol_issue", "answer_only",
        "format_issue", "outcome", "attempts", "retry_reasons",
        "truncated", "final_words", "final_answer",
    ],
)

protocol_failures = [row for row in protocol_rows if row["protocol_ok"] is False]
format_failures = [row for row in protocol_rows if not row["answer_only"]]
print("thinking protocol failures:", len(protocol_failures))
if protocol_failures:
    display_records(
        protocol_failures,
        ["model", "case", "mode", "protocol_issue", "truncated", "raw_preview"],
    )
print("final-answer format failures:", len(format_failures))
if format_failures:
    display_records(
        format_failures,
        ["model", "case", "mode", "format_issue", "final_answer"],
    )


In [ ]:
retry_rows = []
for response in responses:
    if response.generation_attempts <= 1:
        continue
    for attempt in response.attempt_history:
        retry_rows.append(
            {
                "model": response.model_id.rsplit("/", 1)[-1],
                "case": response.case_id,
                "mode": response.mode,
                "attempt": attempt["attempt"],
                "seed": attempt["seed"],
                "label": attempt["answer_label"],
                "protocol_issue": attempt["thinking_protocol_issue"],
                "retry_reason": attempt["retry_reason"],
                "retried": attempt["retried"],
                "answer": attempt["answer"].replace("\n", " ")[:160],
            }
        )

print("responses regenerated:", sum(response.generation_attempts > 1 for response in responses))
if retry_rows:
    display_records(
        retry_rows,
        [
            "model", "case", "mode", "attempt", "seed", "label",
            "protocol_issue", "retry_reason", "retried", "answer",
        ],
    )


## 결과를 읽는 순서

1. 먼저 `truncated`가 없는지 확인합니다. 잘린 응답은 정오 판정에서 제외합니다.
2. thinking 응답은 `think_protocol_ok=True`여야 하며, 실패했다면 `protocol_issue`와 `raw_preview`로 원인을 확인합니다.
3. non-thinking 응답은 thinking tag가 없어야 합니다. `answer_only=False`는 별개의 최종 출력 형식 실패입니다.
4. `truncated_before_think_close`는 출력 길이 문제이고, `missing_thinking_block`은 모델의 protocol 불이행이므로 같은 실패로 합치지 않습니다.
5. `both`는 정답과 함정 답을 함께 언급한 응답입니다. 다른 seed로 재생성하며, 재시도 전후 원문은 JSON의 `attempt_history`에서 비교합니다.
6. 같은 모델의 thinking/non-thinking 차이를 비교해 reasoning이 함정 답을 수정하는지 확인합니다.
7. 모델 크기별 결과는 합쳐서 하나의 정확도로 만들지 말고, 2B/9B/27B/35B-A3B를 각각 보고합니다.
8. 이후 feature 실험은 exact SAE가 있는 `Qwen3.5-27B`를 주 대상으로 삼고, 다른 크기는 대응 Base checkpoint에서 재현성을 확인합니다.
9. 마지막 그래프의 `Hallucination / other`는 `both`와 `other`를 합친 운영상 범주입니다. 형식 실패나 유효한 다른 표현도 섞일 수 있으므로 순수한 사실 환각으로 단정하지 말고 품질 표와 Markdown 리포트를 함께 확인합니다.
10. CRT 결과는 `crt_difference`, `crt_rate`, `crt_growth`를 합치지 말고 유형별 lure rate와 Wilson 95% 구간을 확인합니다.
11. non-thinking에서 lure가 거의 없다는 결과도 중요합니다. 다만 non-thinking은 reasoning text를 출력하지 않는 모드이지 내부 계산이 없는 System 1 조건과 같지 않습니다.
12. seed 하나의 Wilson 구간은 문항별 이항 변동을 보여주는 기술 통계입니다. 여러 seed를 합칠 때는 같은 문항의 반복을 독립 표본으로 보지 말고 item-clustered bootstrap을 사용합니다.
13. `hagendorff_smoke`를 통과한 뒤 `hagendorff_full`과 여러 `SEEDS`로 본 실행하고, 공개 문항 오염 가능성 때문에 최종 결론은 별도 held-out 변형에서 재검증합니다.

In [ ]:
import matplotlib.pyplot as plt

if not accuracy_rows:
    raise RuntimeError("No responses to summarize. Run the generation cell first.")

chart_labels = [f"{row['model']} | {row['mode']}" for row in accuracy_rows]
correct_counts = [row["correct"] for row in accuracy_rows]
lure_counts = [row["lure"] for row in accuracy_rows]
hallucination_counts = [row["hallucination"] for row in accuracy_rows]
positions = list(range(len(accuracy_rows)))
max_total = max(row["total"] for row in accuracy_rows)

fig_height = max(4.5, 0.72 * len(accuracy_rows))
fig, ax = plt.subplots(figsize=(11, fig_height), constrained_layout=True)
ax.barh(positions, correct_counts, color="#009E73", label="Correct")
ax.barh(
    positions,
    lure_counts,
    left=correct_counts,
    color="#D55E00",
    label="Lure answer",
)
ax.barh(
    positions,
    hallucination_counts,
    left=[correct + lure for correct, lure in zip(correct_counts, lure_counts)],
    color="#0072B2",
    label="Hallucination / other",
)

for position, row in zip(positions, accuracy_rows, strict=True):
    if row["correct"]:
        ax.text(
            row["correct"] / 2, position, str(row["correct"]),
            ha="center", va="center", color="white", fontweight="bold",
        )
    if row["lure"]:
        ax.text(
            row["correct"] + row["lure"] / 2,
            position,
            str(row["lure"]),
            ha="center", va="center", color="white", fontweight="bold",
        )
    if row["hallucination"]:
        ax.text(
            row["correct"] + row["lure"] + row["hallucination"] / 2,
            position,
            str(row["hallucination"]),
            ha="center", va="center", color="white", fontweight="bold",
        )
    ax.text(
        row["total"] + max_total * 0.02,
        position,
        f"{row['accuracy']:.0%}",
        va="center",
    )

ax.set_yticks(positions, chart_labels)
ax.invert_yaxis()
ax.set_xlim(0, max_total * 1.15)
ax.set_xlabel("Number of CRT questions")
ax.set_title("CRT accuracy by model and reasoning mode")
ax.grid(axis="x", alpha=0.2)
ax.legend(frameon=False, loc="lower right")
plt.show()

families = list(dict.fromkeys(row["family"] for row in family_accuracy_rows))
if families:
    fig, axes = plt.subplots(
        1,
        len(families),
        figsize=(max(7, 4.6 * len(families)), 5),
        sharey=True,
        constrained_layout=True,
    )
    if len(families) == 1:
        axes = [axes]

    for ax, family in zip(axes, families, strict=True):
        rows = [row for row in family_accuracy_rows if row["family"] == family]
        labels = [f"{row['model']}\n{row['mode']}" for row in rows]
        rates = [row["lure_rate"] for row in rows]
        lower = [row["lure_rate"] - row["lure_rate_ci_low"] for row in rows]
        upper = [row["lure_rate_ci_high"] - row["lure_rate"] for row in rows]
        positions = list(range(len(rows)))
        ax.bar(
            positions,
            rates,
            yerr=[lower, upper],
            capsize=3,
            color="#D55E00",
            alpha=0.85,
        )
        ax.set_xticks(positions, labels, rotation=55, ha="right")
        ax.set_ylim(0, 1)
        ax.set_title(family)
        ax.grid(axis="y", alpha=0.2)
    axes[0].set_ylabel("Lure-answer rate (Wilson 95% CI)")
    fig.suptitle("Nature CRT lure rate by family, model, and reasoning mode")
    plt.show()

print(f"Readable report: {REPORT_PATH}")
print("Hallucination / other = unresolved both + other; audit the quality table before interpretation.")
